In [ ]:
import os                           # Used to access environment variables
import numpy as np                  # Used for displaying embeddings
from langchain_community.document_loaders import CSVLoader     # Loads data from a CSV file
from langchain_text_splitters import CharacterTextSplitter     # Splits large text into smaller chunks
from langchain_openai import OpenAIEmbeddings, ChatOpenAI       # OpenAI embedding model and LLM
from langchain_community.vectorstores import FAISS              # FAISS vector database
from langchain_classic.chains import RetrievalQA                 # Retrieval-Augmented Generation (RAG) chain
from langchain_classic.prompts import PromptTemplate              # Used to create custom prompts

In [ ]:
# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------

# Maximum number of characters in each text chunk
CHUNK_SIZE = 1000

# Number of overlapping characters between consecutive chunks
CHUNK_OVERLAP = 100

# Maximum number of tokens the LLM can generate in its response
MAX_TOKENS = 150

# OpenAI model to use
MODEL_NAME = "gpt-4o-mini"      # Change to another model if required

# Controls randomness (lower value = more deterministic responses)
TEMPERATURE = 0.4

from dotenv import load_dotenv
from openai import OpenAI
import os

# Load environment variables from .env
load_dotenv(dotenv_path="../.env")
api_key = os.getenv("OPENAI_API_KEY")
OpenAI.api_key = api_key

In [ ]:
# ---------------------------------------------------
# Load CSV Data
# ---------------------------------------------------

def load_csv_data(file_path):
    """Loads the CSV file into LangChain Documents."""

    # Create CSV loader
    loader = CSVLoader(file_path=file_path)

    # Read all rows from the CSV
    documents = loader.load()

    # Display number of documents loaded
    print(f"Loaded {len(documents)} documents from CSV")

    return documents

In [ ]:
# ----------------------------------------------------------
# Split Documents and Generate Embeddings
# ----------------------------------------------------------

def create_embeddings(documents):
    """Splits documents into chunks and creates embeddings."""

    # Create text splitter
    text_splitter = CharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP
    )

    # Split documents into smaller chunks
    texts = text_splitter.split_documents(documents)

    print(f"Split into {len(texts)} chunks")

    # Create embedding model
    embeddings = OpenAIEmbeddings()

    # Display one sample embedding for learning purposes
    if texts:

        # Get the first text chunk
        sample_text = texts[0].page_content

        # Convert text into an embedding vector
        sample_embedding = embeddings.embed_query(sample_text)

        print("\nSample Text:")
        print(sample_text)

        print("\nSample Embedding (first 10 dimensions):")
        print(np.array(sample_embedding[:10]))

        print(f"\nEmbedding shape: {np.array(sample_embedding).shape}")

    return texts, embeddings

In [ ]:
# ----------------------------------------------------------
# Create FAISS Vector Store
# ----------------------------------------------------------

def create_vectorstore(texts, embeddings):
    """Stores document embeddings inside FAISS."""

    # Create FAISS vector database
    vectorstore = FAISS.from_documents(
        texts,
        embeddings
    )

    return vectorstore

In [ ]:
# ----------------------------------------------------------
# Create RetrievalQA Chain
# ----------------------------------------------------------

def setup_qa_chain(vectorstore):

    # Initialize the language model
    llm = ChatOpenAI(
        model_name=MODEL_NAME,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS
    )

    # Prompt given to the LLM
    template = """
Use the following pieces of context to answer the question at the end.

If you don't know the answer,
just say that you don't know.

Don't try to make up an answer.

{context}

Question: {question}

Answer:
"""

    # Create a prompt template
    PROMPT = PromptTemplate(
        template=template,
        input_variables=["context", "question"]
    )

    # Create the Retrieval-Augmented Generation (RAG) chain
    qa_chain = RetrievalQA.from_chain_type(

        # Language model used to generate answers
        llm=llm,

        # "stuff" places all retrieved documents into one prompt
        chain_type="stuff",

        # Retriever searches the FAISS vector database
        retriever=vectorstore.as_retriever(),

        # Return the retrieved source documents
        return_source_documents=True,

        # Use the custom prompt
        chain_type_kwargs={
            "prompt": PROMPT
        }
    )

    return qa_chain


In [ ]:
# ----------------------------------------------------------
# Ask a Question
# ----------------------------------------------------------

def process_query(query, qa_chain):

    # Send the user's question to the RetrievalQA chain
    result = qa_chain({
        "query": query
    })

    # Return both the generated answer and the retrieved documents
    return result["result"], result["source_documents"]

In [ ]:
print("Welcome to the CSV RAG Pipeline!")

# Ask the user for the CSV file path
csv_path = input("Please enter the path to your CSV file: ")

print("Loading and processing CSV data...")

# Load the CSV documents
documents = load_csv_data(csv_path)

print("Creating embeddings...")

# Split documents and create embeddings
texts, embeddings = create_embeddings(documents)

print("Creating vector store...")

# Store embeddings in FAISS
vectorstore = create_vectorstore(texts, embeddings)

print("Setting up QA chain...")

# Build the RetrievalQA pipeline
qa_chain = setup_qa_chain(vectorstore)

print("\nRAG Pipeline initialized.")
print("You can now ask questions about your CSV data.")
print("Enter 'quit' to exit the program.")

# Keep accepting questions until the user quits
while True:

    # Read the user's question
    query = input("\nEnter your question: ")

    # Exit if the user types "quit"
    if query.lower() == "quit":
        print("Exiting the program. Goodbye!")
        break

    # Retrieve the answer and source documents
    answer, sources = process_query(query, qa_chain)

    # Display the generated answer
    print(f"\nAnswer: {answer}")

    # Display the source documents used to generate the answer
    print("\nSources:")

    for source in sources:
        print(f"- {source.page_content[:100]}...")